# BEST-Rec v2: Rigorous Re-implementation

## BERT-Embedded Self-Attention Transformer Recommender

### Key fixes over v1:
| Issue | v1 (broken) | v2 (fixed) |
|---|---|---|
| SVD leakage | SVD on full data before split | SVD recomputed per fold on train only |
| User embed leakage | All reviews embedded before split | Reviews embedded per fold from train only |
| CV split | `KFold` on interactions | `GroupKFold` by `user_id` |
| Cross-attention | Single token → trivial `softmax=1` | Multi-token (5 user × 3 item tokens) |
| Bidirectional | Only user→item | user→item AND item→user |
| Cold-start eval | Not actually evaluated | Held-out users/items, separate protocol |
| Ranking metrics | None | NDCG@10, HR@10, Recall@10 |
| Model checkpointing | Not saved | Best model restored after early stop |
| Baselines | Cited from other papers | Run locally under identical splits |
| Datasets | 2 (Beauty, Books) | 4 (+Fashion, Instruments) |

## 0. Install Dependencies

Run this cell once. It installs everything needed into the current kernel. Restart the kernel after if prompted.

In [1]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# PyTorch with CUDA 12.8 (for RTX 50-series / Blackwell)
pip_install("torch", "torchvision", "torchaudio",
            "--index-url", "https://download.pytorch.org/whl/cu128")

# ML & NLP dependencies
pip_install("transformers", "scikit-learn", "scipy", "numpy",
            "tqdm", "pandas", "ipywidgets")

print("All dependencies installed.")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3.12 install --upgrade pip


All dependencies installed.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3.12 install --upgrade pip


## 1. Imports & Environment Setup

In [2]:
import os
import sys
import json
import math
import pickle
import copy
import time
import hashlib
import warnings
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import GroupKFold
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.sparse import csr_matrix, coo_matrix

from transformers import AutoTokenizer, AutoModel

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── CUDA optimisations ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:  {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    gpu_name = "CPU"
    vram_gb  = 0
    print("Warning: No CUDA GPU detected — running on CPU (will be slow)")

print(f"CPU cores: {os.cpu_count()}")
print(f"PyTorch:   {torch.__version__}")
print(f"Python:    {sys.version.split()[0]}")

GPU:  NVIDIA GeForce RTX 5060 Ti
VRAM: 17.1 GB
CPU cores: 20
PyTorch:   2.11.0+cu128
Python:    3.12.10


## 2. Configuration

All hyperparameters are centralised here. Change `DATASET` to run on different Amazon categories.

**Cache strategy**: Every expensive computation (data parsing, text embeddings, SVD, etc.) is cached to disk. Re-running a cell skips the computation if the cache exists. Delete the `cache/` directory to force a full recompute.

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CHANGE THIS to run on a different dataset
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATASET = "beauty"   # "beauty" | "books" | "fashion" | "instruments"

# ── Dataset file mapping (Amazon Reviews 2023) ──────
DATASET_FILES = {
    "beauty":      ("All_Beauty.jsonl",          "meta_All_Beauty.jsonl"),
    "books":       ("Books.jsonl",               "meta_Books.jsonl"),
    "fashion":     ("Amazon_Fashion.jsonl",      "meta_Amazon_Fashion.jsonl"),
    "instruments": ("Musical_Instruments.jsonl", "meta_Musical_Instruments.jsonl"),
}

# ── Paths ────────────────────────────────────────────
DATA_DIR   = "./data"
CACHE_DIR  = f"./cache/{DATASET}"
os.makedirs(CACHE_DIR, exist_ok=True)

INTER_FILE, META_FILE = DATASET_FILES[DATASET]
INTER_PATH = os.path.join(DATA_DIR, DATASET, INTER_FILE)
META_PATH  = os.path.join(DATA_DIR, DATASET, META_FILE)

# ── Hardware-aware settings ──────────────────────────
NUM_CPU_WORKERS = min(8, max(0, os.cpu_count() - 2))
USE_AMP         = device.type == "cuda"
USE_COMPILE     = hasattr(torch, "compile")
PIN_MEMORY      = device.type == "cuda"
PREFETCH_FACTOR = 4

# Scale batch size to available VRAM
if vram_gb >= 12:
    BATCH_SIZE = 4096
elif vram_gb >= 8:
    BATCH_SIZE = 2048
else:
    BATCH_SIZE = 1024

# ── Model hyperparameters ────────────────────────────
PRETRAINED_MODEL   = "distilbert-base-uncased"
TEXT_DIM           = 768
HIDDEN_DIM         = 512
NUM_HEADS          = 4
NUM_ENCODER_LAYERS = 2
MAX_USER_REVIEWS   = 5
SVD_COMPONENTS     = 512
NUM_CLASSES        = 5

# ── Training hyperparameters ─────────────────────────
LR           = 1e-4
EPOCHS       = 30
PATIENCE     = 7
LAMBDA_CLS   = 1.5
WEIGHT_DECAY = 1e-5
GRAD_CLIP    = 1.0

# ── Evaluation settings ─────────────────────────────
NUM_FOLDS            = 5
NEG_SAMPLES          = 99
TOP_K                = 10
COLD_USER_THRESHOLD  = 3
COLD_ITEM_THRESHOLD  = 5
RANKING_EVAL_USERS   = 2000

# ── Reproducibility ──────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

print(f"Dataset:     {DATASET}")
print(f"Batch size:  {BATCH_SIZE}  (auto-scaled to {vram_gb:.0f} GB VRAM)")
print(f"Workers:     {NUM_CPU_WORKERS}")
print(f"AMP (FP16):  {USE_AMP}")
print(f"Compile:     {USE_COMPILE}")
print(f"Config:      H={HIDDEN_DIM}, SVD_k={SVD_COMPONENTS}, LR={LR}")

Dataset:     beauty
Batch size:  4096  (auto-scaled to 17 GB VRAM)
Workers:     8
AMP (FP16):  True
Compile:     True
Config:      H=512, SVD_k=512, LR=0.0001


## 3. Cache Utility

Wraps any expensive function: if the cache file exists, load it; otherwise compute and save.

In [4]:
def cached(cache_name: str, compute_fn, force_recompute=False):
    path = os.path.join(CACHE_DIR, cache_name)
    if os.path.exists(path) and not force_recompute:
        print(f"  Cache hit: {cache_name}")
        with open(path, "rb") as f:
            return pickle.load(f)
    print(f"  Cache miss: {cache_name} — computing...")
    result = compute_fn()
    with open(path, "wb") as f:
        pickle.dump(result, f)
    print(f"  Saved: {cache_name} ({os.path.getsize(path)/1e6:.1f} MB)")
    return result


def cached_tensor(cache_name: str, compute_fn, force_recompute=False):
    path = os.path.join(CACHE_DIR, cache_name)
    if os.path.exists(path) and not force_recompute:
        print(f"  Cache hit: {cache_name}")
        return torch.load(path, weights_only=True)
    print(f"  Cache miss: {cache_name} — computing...")
    result = compute_fn()
    torch.save(result, path)
    print(f"  Saved: {cache_name} ({os.path.getsize(path)/1e6:.1f} MB)")
    return result

## 4. Data Loading

Parse the raw JSONL files into structured interactions and item metadata. **Cached.**

In [5]:
def _load_raw_data():
    user2id, item2id = {}, {}
    interactions = []
    errors = 0

    print(f"  Reading interactions: {INTER_PATH}")
    with open(INTER_PATH, "r") as f:
        for line in tqdm(f, desc="Interactions", unit="line"):
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                errors += 1
                continue
            uid_raw = rec.get("user_id")
            iid_raw = rec.get("parent_asin")
            rating  = rec.get("rating")
            if uid_raw is None or iid_raw is None or rating is None:
                continue

            review_text  = rec.get("text", "") or ""
            review_title = rec.get("title", "") or ""
            full_review  = (review_title + " " + review_text).strip()

            if uid_raw not in user2id:
                user2id[uid_raw] = len(user2id)
            if iid_raw not in item2id:
                item2id[iid_raw] = len(item2id)

            interactions.append({
                "user_id": user2id[uid_raw],
                "item_id": item2id[iid_raw],
                "rating":  float(rating),
                "review":  full_review,
            })

    print(f"  Reading metadata: {META_PATH}")
    item_metadata = {}
    with open(META_PATH, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Metadata", unit="line"):
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            pasin = rec.get("parent_asin")
            if pasin not in item2id:
                continue
            idx = item2id[pasin]
            price = rec.get("price", 0.0)
            item_metadata[idx] = {
                "title":      rec.get("title", "") or "",
                "avg_rating": float(rec.get("average_rating", 0.0) or 0.0),
                "rating_num": float(rec.get("rating_number", 0.0) or 0.0),
                "price":      float(price) if price else 0.0,
            }

    for i in range(len(item2id)):
        if i not in item_metadata:
            item_metadata[i] = {"title": "", "avg_rating": 0.0,
                                "rating_num": 0.0, "price": 0.0}

    return {
        "interactions":  interactions,
        "user2id":       user2id,
        "item2id":       item2id,
        "item_metadata": item_metadata,
        "num_users":     len(user2id),
        "num_items":     len(item2id),
    }


data = cached("raw_data.pkl", _load_raw_data)

interactions  = data["interactions"]
item_metadata = data["item_metadata"]
num_users     = data["num_users"]
num_items     = data["num_items"]

print(f"\nUsers: {num_users:,}   Items: {num_items:,}   Interactions: {len(interactions):,}")

  Cache hit: raw_data.pkl

Users: 631,986   Items: 112,565   Interactions: 701,528


## 5. Dataset Statistics

In [6]:
density = len(interactions) / (num_users * num_items)
user_counts = defaultdict(int)
item_counts = defaultdict(int)
ratings = []

for inter in interactions:
    user_counts[inter["user_id"]] += 1
    item_counts[inter["item_id"]] += 1
    ratings.append(inter["rating"])

cold_users = sum(1 for c in user_counts.values() if c <= COLD_USER_THRESHOLD)
cold_items = sum(1 for c in item_counts.values() if c <= COLD_ITEM_THRESHOLD)
ratings = np.array(ratings)

print(f"Dataset:       {DATASET}")
print(f"Users:         {num_users:>12,}")
print(f"Items:         {num_items:>12,}")
print(f"Interactions:  {len(interactions):>12,}")
print(f"Density:       {density:>12.8f}")
print(f"Sparsity:      {1-density:>12.8f}")
print(f"Cold users (le{COLD_USER_THRESHOLD}):  {cold_users:>10,}  ({100*cold_users/num_users:.1f}%)")
print(f"Cold items (le{COLD_ITEM_THRESHOLD}):  {cold_items:>10,}  ({100*cold_items/num_items:.1f}%)")
print(f"Rating mean:   {ratings.mean():.3f}")
print(f"Rating std:    {ratings.std():.3f}")
for r in [1,2,3,4,5]:
    print(f"  Rating {r}: {int((ratings==r).sum()):>10,}")

Dataset:       beauty
Users:              631,986
Items:              112,565
Interactions:       701,528
Density:         0.00000986
Sparsity:        0.99999014
Cold users (le3):     628,540  (99.5%)
Cold items (le5):      89,699  (79.7%)
Rating mean:   3.960
Rating std:    1.494
  Rating 1:    102,080
  Rating 2:     43,034
  Rating 3:     56,307
  Rating 4:     79,381
  Rating 5:    420,726


## 6. Text Encoder (DistilBERT)

**Item title embeddings** are precomputed once and cached — they come from item metadata (not interaction data), so there is **no leakage**.

**User review embeddings** are computed **per fold** from training-only interactions to prevent leakage. Each user gets up to `MAX_USER_REVIEWS` separate review embeddings (multi-token).

In [7]:
class TextEncoder:
    def __init__(self, model_name, device, max_length=64):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()
        self.device = device
        self.max_length = max_length
        self.dim = self.model.config.hidden_size

        if USE_COMPILE:
            try:
                self.model = torch.compile(self.model, mode="reduce-overhead")
                print("  DistilBERT compiled with torch.compile")
            except Exception:
                print("  torch.compile failed for DistilBERT, using eager mode")

    @torch.no_grad()
    def encode_batch(self, texts: List[str], batch_size=256) -> torch.Tensor:
        all_embeds = []
        for i in range(0, len(texts), batch_size):
            batch = [t if t.strip() else "empty" for t in texts[i:i+batch_size]]
            inputs = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=self.max_length, return_tensors="pt"
            ).to(self.device)
            with autocast(enabled=USE_AMP):
                outputs = self.model(**inputs)
            cls = outputs.last_hidden_state[:, 0, :].float()
            all_embeds.append(cls.cpu())
        return torch.cat(all_embeds, dim=0)

    def encode_item_titles(self, item_metadata, num_items):
        titles = [item_metadata[i]["title"] for i in range(num_items)]
        print(f"  Encoding {len(titles):,} item titles...")
        return self.encode_batch(titles)

    def encode_user_reviews_for_fold(self, train_interactions, num_users, max_reviews=5):
        print(f"  Encoding user reviews (train-only, max {max_reviews} per user)...")

        user_reviews = defaultdict(list)
        for inter in train_interactions:
            if inter["review"].strip():
                user_reviews[inter["user_id"]].append(inter["review"])

        uid_slot_text = []
        for uid in range(num_users):
            for j, rev in enumerate(user_reviews.get(uid, [])[:max_reviews]):
                uid_slot_text.append((uid, j, rev))

        embeds = torch.zeros(num_users, max_reviews, self.dim)
        masks  = torch.zeros(num_users, max_reviews, dtype=torch.bool)

        if uid_slot_text:
            texts = [t[2] for t in uid_slot_text]
            print(f"    {len(texts):,} reviews to encode...")
            encoded = self.encode_batch(texts)
            for idx, (uid, j, _) in enumerate(uid_slot_text):
                embeds[uid, j] = encoded[idx]
                masks[uid, j]  = True

        n_with = masks.any(dim=1).sum().item()
        print(f"    {n_with:,}/{num_users:,} users have reviews")
        return embeds, masks


text_encoder = TextEncoder(PRETRAINED_MODEL, device)
print(f"TextEncoder ready (dim={text_encoder.dim})")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  DistilBERT compiled with torch.compile
TextEncoder ready (dim=768)


### 6a. Precompute Item Title Embeddings (cached, no leakage)

In [8]:
item_title_embeds = cached_tensor(
    "item_title_embeds.pt",
    lambda: text_encoder.encode_item_titles(item_metadata, num_items)
)
print(f"Item title embeddings: {item_title_embeds.shape}")

  Cache hit: item_title_embeds.pt
Item title embeddings: torch.Size([112565, 768])


### 6b. Precompute Numeric Features (cached, no leakage)

In [9]:
def _compute_numeric():
    avg_r = torch.tensor([item_metadata[i]["avg_rating"] for i in range(num_items)])
    r_num = torch.tensor([item_metadata[i]["rating_num"] for i in range(num_items)])
    price = torch.tensor([item_metadata[i]["price"]      for i in range(num_items)])
    def zscore(x):
        return (x - x.mean()) / (x.std() + 1e-9)
    return torch.stack([zscore(avg_r), zscore(r_num), zscore(price)], dim=-1)

item_numeric = cached_tensor("item_numeric.pt", _compute_numeric)
print(f"Item numeric features: {item_numeric.shape}")

  Cache hit: item_numeric.pt
Item numeric features: torch.Size([112565, 3])


## 7. Per-Fold Feature Engineering (NO LEAKAGE)

These are called **inside each fold** with only training interactions. Both are cached with fold-specific keys.

In [10]:
def compute_svd_for_fold(train_inters, fold_tag, force=False):
    cache_name = f"svd_fold_{fold_tag}.pt"
    def _compute():
        rows, cols, vals = [], [], []
        for inter in train_inters:
            rows.append(inter["item_id"])
            cols.append(inter["user_id"])
            vals.append(inter["rating"])
        mat = csr_matrix((vals, (rows, cols)), shape=(num_items, num_users))
        k = min(SVD_COMPONENTS, min(num_items, num_users) - 1, len(set(rows)) - 1)
        if k < 1:
            return torch.zeros(num_items, SVD_COMPONENTS)
        svd = TruncatedSVD(n_components=k, random_state=SEED)
        result = svd.fit_transform(mat)
        explained = svd.explained_variance_ratio_.sum()
        print(f"    SVD: k={k}, explained variance={explained:.4f}")
        if k < SVD_COMPONENTS:
            result = np.hstack([result, np.zeros((num_items, SVD_COMPONENTS - k))])
        return torch.tensor(result, dtype=torch.float32)
    return cached_tensor(cache_name, _compute, force_recompute=force)


def compute_user_embeds_for_fold(train_inters, fold_tag, force=False):
    cache_name = f"user_embeds_fold_{fold_tag}.pkl"
    def _compute():
        embeds, masks = text_encoder.encode_user_reviews_for_fold(
            train_inters, num_users, MAX_USER_REVIEWS)
        return {"embeds": embeds, "masks": masks}
    result = cached(cache_name, _compute, force_recompute=force)
    return result["embeds"], result["masks"]

## 8. Dataset Class (Multi-Token)

Pre-indexes all tensors for zero-copy `__getitem__`. All heavy tensors stay in shared CPU memory for fast worker access.

In [11]:
class BESTRecDataset(Dataset):

    def __init__(self, interactions, user_review_embeds, user_masks,
                 item_title_embeds, item_numeric, item_svd):
        self.user_ids   = torch.tensor([i["user_id"] for i in interactions], dtype=torch.long)
        self.item_ids   = torch.tensor([i["item_id"] for i in interactions], dtype=torch.long)
        self.ratings    = torch.tensor([i["rating"]  for i in interactions], dtype=torch.float32)
        self.cls_labels = torch.clamp(self.ratings.long() - 1, min=0)

        self.user_review_embeds = user_review_embeds
        self.user_masks         = user_masks
        self.item_title_embeds  = item_title_embeds
        self.item_numeric       = item_numeric
        self.item_svd           = item_svd

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        uid = self.user_ids[idx]
        iid = self.item_ids[idx]
        return (
            self.user_review_embeds[uid],
            self.user_masks[uid],
            self.item_title_embeds[iid],
            self.item_numeric[iid],
            self.item_svd[iid],
            self.ratings[idx],
            self.cls_labels[idx],
            uid,
            iid,
        )

## 9. BEST-Rec v2 Model Architecture

**Multi-token, truly bidirectional cross-attention.**

| Side | Tokens | Source |
|---|---|---|
| User | up to 5 | Individual review [CLS] embeddings |
| Item | 3 | Title embed, Numeric embed, SVD embed |

Cross-attention operates on real sequences — `softmax(QK^T / sqrt(d))` distributes genuine attention weights across multiple keys.

In [12]:
class BESTRecV2(nn.Module):

    def __init__(self, text_dim=TEXT_DIM, hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS,
                 num_layers=NUM_ENCODER_LAYERS, svd_dim=SVD_COMPONENTS,
                 num_classes=NUM_CLASSES, use_text=True, use_numeric=True,
                 use_svd=True, use_cross_attn=True, bidirectional=True):
        super().__init__()
        H = hidden_dim
        self.use_text       = use_text
        self.use_numeric    = use_numeric
        self.use_svd        = use_svd
        self.use_cross_attn = use_cross_attn
        self.bidirectional  = bidirectional

        self.user_review_proj  = nn.Linear(text_dim, H)
        self.item_title_proj   = nn.Linear(text_dim, H)
        self.item_numeric_proj = nn.Linear(3, H)
        self.item_svd_proj     = nn.Linear(svd_dim, H)

        self.item_token_type = nn.Embedding(3, H)

        u_layer = nn.TransformerEncoderLayer(
            d_model=H, nhead=num_heads, dim_feedforward=H*4,
            dropout=0.1, batch_first=True, activation="gelu")
        self.user_encoder = nn.TransformerEncoder(u_layer, num_layers=num_layers)

        i_layer = nn.TransformerEncoderLayer(
            d_model=H, nhead=num_heads, dim_feedforward=H*4,
            dropout=0.1, batch_first=True, activation="gelu")
        self.item_encoder = nn.TransformerEncoder(i_layer, num_layers=num_layers)

        self.cross_attn_u2i = nn.MultiheadAttention(
            embed_dim=H, num_heads=num_heads, batch_first=True, dropout=0.1)
        self.norm_u2i = nn.LayerNorm(H)

        if bidirectional:
            self.cross_attn_i2u = nn.MultiheadAttention(
                embed_dim=H, num_heads=num_heads, batch_first=True, dropout=0.1)
            self.norm_i2u = nn.LayerNorm(H)
            fusion_in = H * 4
        else:
            fusion_in = H * 2

        if not use_cross_attn:
            fusion_in = H * 2

        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, H), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(H, H),         nn.GELU(), nn.Dropout(0.1))

        self.regressor = nn.Sequential(
            nn.Linear(H, H // 2), nn.GELU(), nn.Linear(H // 2, 1))
        self.classifier = nn.Linear(H, num_classes)

    def forward(self, user_reviews, user_mask, item_title, item_numeric, item_svd):
        B = user_reviews.size(0)

        u_tokens   = self.user_review_proj(user_reviews)
        u_pad_mask = ~user_mask
        all_pad    = u_pad_mask.all(dim=1)
        if all_pad.any():
            u_pad_mask[all_pad, 0] = False

        u_enc = self.user_encoder(u_tokens, src_key_padding_mask=u_pad_mask)

        tokens_list, type_ids = [], []
        if self.use_text:
            tokens_list.append(self.item_title_proj(item_title).unsqueeze(1))
            type_ids.append(0)
        if self.use_numeric:
            tokens_list.append(self.item_numeric_proj(item_numeric).unsqueeze(1))
            type_ids.append(1)
        if self.use_svd:
            tokens_list.append(self.item_svd_proj(item_svd).unsqueeze(1))
            type_ids.append(2)
        if not tokens_list:
            tokens_list.append(self.item_title_proj(item_title).unsqueeze(1))
            type_ids.append(0)

        i_tokens = torch.cat(tokens_list, dim=1)
        type_t   = torch.tensor(type_ids, device=i_tokens.device).unsqueeze(0).expand(B, -1)
        i_tokens = i_tokens + self.item_token_type(type_t)
        i_enc    = self.item_encoder(i_tokens)

        u_mask_f = user_mask.unsqueeze(-1).float()
        u_pool   = (u_enc * u_mask_f).sum(1) / (u_mask_f.sum(1) + 1e-9)
        i_pool   = i_enc.mean(dim=1)

        if self.use_cross_attn:
            u2i, _ = self.cross_attn_u2i(query=u_enc, key=i_enc, value=i_enc)
            u2i    = self.norm_u2i(u2i + u_enc)
            u2i_pool = (u2i * u_mask_f).sum(1) / (u_mask_f.sum(1) + 1e-9)

            if self.bidirectional:
                i2u, _ = self.cross_attn_i2u(
                    query=i_enc, key=u_enc, value=u_enc,
                    key_padding_mask=u_pad_mask)
                i2u = self.norm_i2u(i2u + i_enc)
                i2u_pool = i2u.mean(dim=1)
                fused_in = torch.cat([u_pool, u2i_pool, i_pool, i2u_pool], dim=-1)
            else:
                fused_in = torch.cat([u_pool, u2i_pool], dim=-1)
        else:
            fused_in = torch.cat([u_pool, i_pool], dim=-1)

        z = self.fusion(fused_in)
        rating_pred = self.regressor(z).squeeze(-1)
        cls_logits  = self.classifier(z)
        return rating_pred, cls_logits

## 10. Baseline Models

Run under **identical splits** for fair comparison.

In [13]:
class NeuMFBaseline(nn.Module):
    def __init__(self, n_users, n_items, embed_dim=64, hidden=128):
        super().__init__()
        self.user_gmf = nn.Embedding(n_users, embed_dim)
        self.item_gmf = nn.Embedding(n_items, embed_dim)
        self.user_mlp = nn.Embedding(n_users, embed_dim)
        self.item_mlp = nn.Embedding(n_items, embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2),   nn.ReLU())
        self.output = nn.Linear(embed_dim + hidden // 2, 1)

    def forward(self, uids, iids):
        gmf = self.user_gmf(uids) * self.item_gmf(iids)
        mlp_in  = torch.cat([self.user_mlp(uids), self.item_mlp(iids)], dim=-1)
        mlp_out = self.mlp(mlp_in)
        return self.output(torch.cat([gmf, mlp_out], dim=-1)).squeeze(-1)


def run_svd_baseline(train_inters, test_inters, n_components=64):
    rows, cols, vals = [], [], []
    for inter in train_inters:
        rows.append(inter["user_id"])
        cols.append(inter["item_id"])
        vals.append(inter["rating"])
    global_mean = np.mean(vals)
    mat_ui = csr_matrix((vals, (rows, cols)), shape=(num_users, num_items))
    k = min(n_components, min(num_users, num_items) - 1, len(set(rows)) - 1)
    if k < 1:
        return {"mae": 99.0, "rmse": 99.0}
    svd = TruncatedSVD(n_components=k, random_state=SEED)
    user_factors = svd.fit_transform(mat_ui)
    item_factors = svd.components_.T
    preds, targets = [], []
    for inter in test_inters:
        uid, iid = inter["user_id"], inter["item_id"]
        pred = float(user_factors[uid] @ item_factors[iid])
        preds.append(np.clip(pred, 1.0, 5.0))
        targets.append(inter["rating"])
    preds, targets = np.array(preds), np.array(targets)
    return {"mae": mean_absolute_error(targets, preds),
            "rmse": np.sqrt(mean_squared_error(targets, preds))}


def run_neumf_baseline(train_inters, test_inters, epochs=15):
    class SimpleDS(Dataset):
        def __init__(self, inters):
            self.inters = inters
        def __len__(self):
            return len(self.inters)
        def __getitem__(self, i):
            d = self.inters[i]
            return d["user_id"], d["item_id"], float(d["rating"])

    model = NeuMFBaseline(num_users, num_items).to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3)
    crit  = nn.SmoothL1Loss()
    dl_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_CPU_WORKERS,
                 pin_memory=PIN_MEMORY,
                 persistent_workers=True if NUM_CPU_WORKERS > 0 else False)
    train_dl = DataLoader(SimpleDS(train_inters), shuffle=True,  **dl_kw)
    test_dl  = DataLoader(SimpleDS(test_inters),  shuffle=False, **dl_kw)

    for ep in range(epochs):
        model.train()
        for uids, iids, rats in train_dl:
            opt.zero_grad(set_to_none=True)
            pred = model(uids.to(device), iids.to(device))
            loss = crit(pred, rats.float().to(device))
            loss.backward()
            opt.step()

    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for uids, iids, rats in test_dl:
            pred = model(uids.to(device), iids.to(device))
            all_p.append(pred.cpu().numpy())
            all_t.append(rats.numpy())
    preds, targets = np.concatenate(all_p), np.concatenate(all_t)
    return {"mae": mean_absolute_error(targets, preds),
            "rmse": np.sqrt(mean_squared_error(targets, preds))}

print("Baseline models defined.")

Baseline models defined.


## 11. Training & Evaluation Functions

All use **mixed-precision (AMP)** and **non-blocking GPU transfers** for maximum throughput.

In [14]:
amp_scaler = GradScaler(enabled=USE_AMP)


def train_one_epoch(model, optimizer, dataloader):
    global amp_scaler
    model.train()
    crit_reg = nn.SmoothL1Loss()
    crit_cls = nn.CrossEntropyLoss()
    total_loss, n = 0.0, 0

    for batch in tqdm(dataloader, desc="  Train", leave=False):
        u_rev, u_mask, i_tit, i_num, i_svd, rat, cls, _, _ = batch
        u_rev  = u_rev.to(device, non_blocking=True)
        u_mask = u_mask.to(device, non_blocking=True)
        i_tit  = i_tit.to(device, non_blocking=True)
        i_num  = i_num.to(device, non_blocking=True)
        i_svd  = i_svd.to(device, non_blocking=True)
        rat    = rat.to(device, non_blocking=True)
        cls    = cls.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            pred_r, pred_cls = model(u_rev, u_mask, i_tit, i_num, i_svd)
            loss = crit_reg(pred_r, rat) + LAMBDA_CLS * crit_cls(pred_cls, cls)

        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        amp_scaler.step(optimizer)
        amp_scaler.update()

        total_loss += loss.item() * rat.size(0)
        n += rat.size(0)

    return total_loss / n


@torch.no_grad()
def evaluate_rating(model, dataloader):
    model.eval()
    all_preds, all_targets = [], []
    for batch in tqdm(dataloader, desc="  Eval", leave=False):
        u_rev, u_mask, i_tit, i_num, i_svd, rat, _, _, _ = batch
        with autocast(enabled=USE_AMP):
            pred_r, _ = model(
                u_rev.to(device, non_blocking=True),
                u_mask.to(device, non_blocking=True),
                i_tit.to(device, non_blocking=True),
                i_num.to(device, non_blocking=True),
                i_svd.to(device, non_blocking=True))
        all_preds.append(pred_r.float().cpu().numpy())
        all_targets.append(rat.numpy())

    preds   = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
    return mean_absolute_error(targets, preds), np.sqrt(mean_squared_error(targets, preds))


@torch.no_grad()
def evaluate_ranking(model, test_inters, user_review_embeds, user_masks,
                     item_title_embeds, item_numeric, item_svd):
    model.eval()
    user_test_items = defaultdict(set)
    for inter in test_inters:
        user_test_items[inter["user_id"]].add(inter["item_id"])

    ndcg_list, hr_list = [], []
    all_items = set(range(num_items))
    rng = np.random.RandomState(SEED)

    eval_users = [u for u in user_test_items if len(user_test_items[u]) > 0]
    eval_users = eval_users[:RANKING_EVAL_USERS]

    for uid in tqdm(eval_users, desc="  Ranking", leave=False):
        for pos_iid in user_test_items[uid]:
            neg_pool = list(all_items - user_test_items[uid])
            if len(neg_pool) < NEG_SAMPLES:
                continue
            neg_iids = rng.choice(neg_pool, size=NEG_SAMPLES, replace=False)
            cands = [pos_iid] + list(neg_iids)
            n = len(cands)

            u_rev = user_review_embeds[uid].unsqueeze(0).expand(n,-1,-1).to(device)
            u_msk = user_masks[uid].unsqueeze(0).expand(n,-1).to(device)
            i_tit = item_title_embeds[cands].to(device)
            i_num = item_numeric[cands].to(device)
            i_svd = item_svd[cands].to(device)

            with autocast(enabled=USE_AMP):
                scores, _ = model(u_rev, u_msk, i_tit, i_num, i_svd)
            scores = scores.float().cpu().numpy()

            ranked = np.argsort(-scores)
            pos_rank = int(np.where(ranked == 0)[0][0])
            hr_list.append(1.0 if pos_rank < TOP_K else 0.0)
            ndcg_list.append(1.0 / np.log2(pos_rank + 2) if pos_rank < TOP_K else 0.0)

    return {
        f"NDCG@{TOP_K}": np.mean(ndcg_list) if ndcg_list else 0.0,
        f"HR@{TOP_K}":   np.mean(hr_list)   if hr_list   else 0.0,
    }

print("Training & evaluation functions defined (AMP enabled).")

Training & evaluation functions defined (AMP enabled).


## 12. Run a Single Fold

Full pipeline for one train/test split:
1. Compute SVD on training data (cached per fold)
2. Compute user review embeddings on training data (cached per fold)
3. Build datasets and optimised DataLoaders
4. Train with early stopping + **best model checkpointing**
5. Evaluate: rating metrics + ranking metrics

In [15]:
def run_fold(fold_tag, train_inters, test_inters,
             use_text=True, use_numeric=True, use_svd=True,
             use_cross_attn=True, bidirectional=True):

    global amp_scaler
    amp_scaler = GradScaler(enabled=USE_AMP)

    print(f"\n{'='*60}")
    print(f"  Fold [{fold_tag}]: {len(train_inters):,} train / {len(test_inters):,} test")
    print(f"{'='*60}")

    # ── Per-fold SVD (cached) ──
    item_svd = compute_svd_for_fold(train_inters, fold_tag)

    # ── Per-fold user review embeddings (cached) ──
    user_embeds, user_masks = compute_user_embeds_for_fold(train_inters, fold_tag)

    # ── Datasets ──
    train_ds = BESTRecDataset(train_inters, user_embeds, user_masks,
                               item_title_embeds, item_numeric, item_svd)
    test_ds  = BESTRecDataset(test_inters,  user_embeds, user_masks,
                               item_title_embeds, item_numeric, item_svd)

    dl_kwargs = dict(
        batch_size=BATCH_SIZE,
        num_workers=NUM_CPU_WORKERS,
        pin_memory=PIN_MEMORY,
        prefetch_factor=PREFETCH_FACTOR if NUM_CPU_WORKERS > 0 else None,
        persistent_workers=True if NUM_CPU_WORKERS > 0 else False,
    )
    train_dl = DataLoader(train_ds, shuffle=True,  **dl_kwargs)
    test_dl  = DataLoader(test_ds,  shuffle=False, **dl_kwargs)

    # ── Model ──
    model = BESTRecV2(
        use_text=use_text, use_numeric=use_numeric, use_svd=use_svd,
        use_cross_attn=use_cross_attn, bidirectional=bidirectional
    ).to(device)

    # Keep a reference to the uncompiled model for clean checkpointing
    raw_model = model
    if USE_COMPILE:
        try:
            model = torch.compile(model, mode="reduce-overhead")
            print("  Model compiled with torch.compile")
        except Exception as e:
            print(f"  torch.compile skipped: {e}")

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    # ── Training with best-model checkpointing ──
    best_mae   = float("inf")
    best_state = None
    patience_ctr = 0

    for epoch in range(EPOCHS):
        t0 = time.time()
        train_loss = train_one_epoch(model, optimizer, train_dl)
        val_mae, val_rmse = evaluate_rating(model, test_dl)
        scheduler.step(val_mae)
        elapsed = time.time() - t0
        lr_now = optimizer.param_groups[0]["lr"]

        marker = ""
        if val_mae < best_mae:
            best_mae   = val_mae
            best_state = copy.deepcopy(raw_model.state_dict())
            patience_ctr = 0
            marker = " *best*"
        else:
            patience_ctr += 1

        print(f"  Ep {epoch+1:2d}/{EPOCHS} | loss={train_loss:.4f} | "
              f"MAE={val_mae:.4f} | RMSE={val_rmse:.4f} | "
              f"lr={lr_now:.1e} | {elapsed:.0f}s{marker}")

        if patience_ctr >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    # ── Restore best model ──
    raw_model.load_state_dict(best_state)
    print(f"  Restored best model (MAE={best_mae:.4f})")

    # ── Final evaluation ──
    final_mae, final_rmse = evaluate_rating(model, test_dl)
    rank_results = evaluate_ranking(
        model, test_inters, user_embeds, user_masks,
        item_title_embeds, item_numeric, item_svd)

    results = {"mae": final_mae, "rmse": final_rmse, **rank_results}
    print(f"  Results:")
    for k, v in results.items():
        print(f"     {k}: {v:.4f}")

    del model, optimizer, scheduler, train_dl, test_dl
    torch.cuda.empty_cache()

    return results

---
## 13. Experiment 1: Warm Evaluation (GroupKFold by User)

In [ ]:
print("\n" + "=" * 60)
print("  EXPERIMENT 1: Warm Evaluation (GroupKFold by user)")
print("=" * 60)

user_ids_arr = np.array([i["user_id"] for i in interactions])
indices = np.arange(len(interactions))
gkf = GroupKFold(n_splits=NUM_FOLDS)

warm_results = []

for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(indices, groups=user_ids_arr)):
    train_inters = [interactions[i] for i in train_idx]
    test_inters  = [interactions[i] for i in test_idx]
    result = run_fold(f"warm_{fold_idx}", train_inters, test_inters)
    warm_results.append(result)

print("\n" + "=" * 60)
print("WARM EVALUATION SUMMARY")
print("=" * 60)
for metric in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
    vals = [r[metric] for r in warm_results]
    print(f"  {metric:>10s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}   {[round(v,4) for v in vals]}")


  EXPERIMENT 1: Warm Evaluation (GroupKFold by user)

  Fold [warm_0]: 561,222 train / 140,306 test
  Cache hit: svd_fold_warm_0.pt
  Cache hit: user_embeds_fold_warm_0.pkl
  Model compiled with torch.compile


  Train:   0%|          | 0/138 [00:00<?, ?it/s]

W0403 08:44:29.749000 1153386 torch/_functorch/_aot_autograd/autograd_cache.py:1101] [1/1] AOTAutograd cache unable to serialize compiled graph: Please convert all Tensors to FakeTensors first or instantiate FakeTensorMode with 'allow_non_fake_inputs'. Found in aten._to_copy.default(tensor([...], device='cuda:0', size=(24,), dtype=torch.uint8), device=device(type='cpu'))


  Eval:   0%|          | 0/35 [00:00<?, ?it/s]

  Ep  1/30 | loss=1.9748 | MAE=2.5900 | RMSE=2.8764 | lr=1.0e-04 | 30s *best*


...

  Eval:   0%|          | 0/35 [00:00<?, ?it/s]

  Ep 16/30 | loss=1.0736 | MAE=1.7659 | RMSE=1.9179 | lr=5.0e-05 | 16s
  Early stopping at epoch 16
  Restored best model (MAE=1.6459)


  Eval:   0%|          | 0/35 [00:00<?, ?it/s]

  Ranking:   0%|          | 0/2000 [00:00<?, ?it/s]

  Results:
     mae: 1.6459
     rmse: 1.7743
     NDCG@10: 0.0122
     HR@10: 0.0250

  Fold [warm_1]: 561,222 train / 140,306 test
  Cache miss: svd_fold_warm_1.pt — computing...


---
## 14. Experiment 2: Fair Baselines (Same Split)

In [ ]:
print("\n" + "=" * 60)
print("  EXPERIMENT 2: Baselines on fold-0 split")
print("=" * 60)

gkf0 = GroupKFold(n_splits=NUM_FOLDS)
train_idx0, test_idx0 = next(iter(gkf0.split(indices, groups=user_ids_arr)))
train_bl = [interactions[i] for i in train_idx0]
test_bl  = [interactions[i] for i in test_idx0]

baseline_results = {}

print("\n--- SVD Baseline ---")
baseline_results["SVD"] = run_svd_baseline(train_bl, test_bl)
print(f"  SVD:   MAE={baseline_results['SVD']['mae']:.4f}  RMSE={baseline_results['SVD']['rmse']:.4f}")

print("\n--- NeuMF Baseline ---")
baseline_results["NeuMF"] = run_neumf_baseline(train_bl, test_bl)
print(f"  NeuMF: MAE={baseline_results['NeuMF']['mae']:.4f}  RMSE={baseline_results['NeuMF']['rmse']:.4f}")

best_fold0 = warm_results[0]
print("\n--- Comparison (fold 0) ---")
print(f"  {'Model':<12s} {'MAE':>8s} {'RMSE':>8s}")
print(f"  {'-'*30}")
for name, res in baseline_results.items():
    print(f"  {name:<12s} {res['mae']:>8.4f} {res['rmse']:>8.4f}")
print(f"  {'BEST-Rec':<12s} {best_fold0['mae']:>8.4f} {best_fold0['rmse']:>8.4f}")

---
## 15. Experiment 3: Cold-Start Evaluation

**Real cold-start**: users (or items) with very few interactions are **entirely removed from training**.

In [ ]:
def split_cold_users(inters):
    user_c = defaultdict(int)
    for i in inters: user_c[i["user_id"]] += 1
    cold = {u for u, c in user_c.items() if c <= COLD_USER_THRESHOLD}
    train = [i for i in inters if i["user_id"] not in cold]
    test  = [i for i in inters if i["user_id"] in cold]
    print(f"  Cold users: {len(cold):,}  |  Train: {len(train):,}  |  Test: {len(test):,}")
    return train, test

def split_cold_items(inters):
    item_c = defaultdict(int)
    for i in inters: item_c[i["item_id"]] += 1
    cold = {i for i, c in item_c.items() if c <= COLD_ITEM_THRESHOLD}
    train = [i for i in inters if i["item_id"] not in cold]
    test  = [i for i in inters if i["item_id"] in cold]
    print(f"  Cold items: {len(cold):,}  |  Train: {len(train):,}  |  Test: {len(test):,}")
    return train, test


print("\n" + "=" * 60)
print("  EXPERIMENT 3: Cold-Start Evaluation")
print("=" * 60)

# ── Cold-User ──
print("\n--- Cold-User Protocol ---")
cu_train, cu_test = split_cold_users(interactions)
cold_user_result = None
if len(cu_test) >= 10:
    cold_user_result = run_fold("cold_user", cu_train, cu_test)
    print("  Baselines on cold-user split:")
    cu_svd = run_svd_baseline(cu_train, cu_test)
    cu_nmf = run_neumf_baseline(cu_train, cu_test)
    print(f"    SVD:      MAE={cu_svd['mae']:.4f}  RMSE={cu_svd['rmse']:.4f}")
    print(f"    NeuMF:    MAE={cu_nmf['mae']:.4f}  RMSE={cu_nmf['rmse']:.4f}")
    print(f"    BEST-Rec: MAE={cold_user_result['mae']:.4f}  RMSE={cold_user_result['rmse']:.4f}")
else:
    print("  Too few cold-user interactions, skipping.")

# ── Cold-Item ──
print("\n--- Cold-Item Protocol ---")
ci_train, ci_test = split_cold_items(interactions)
cold_item_result = None
if len(ci_test) >= 10:
    cold_item_result = run_fold("cold_item", ci_train, ci_test)
    ci_svd = run_svd_baseline(ci_train, ci_test)
    ci_nmf = run_neumf_baseline(ci_train, ci_test)
    print(f"    SVD:      MAE={ci_svd['mae']:.4f}  RMSE={ci_svd['rmse']:.4f}")
    print(f"    NeuMF:    MAE={ci_nmf['mae']:.4f}  RMSE={ci_nmf['rmse']:.4f}")
    print(f"    BEST-Rec: MAE={cold_item_result['mae']:.4f}  RMSE={cold_item_result['rmse']:.4f}")
else:
    print("  Too few cold-item interactions, skipping.")

---
## 16. Experiment 4: Ablation Study

| Variant | What's changed |
|---|---|
| `full_model` | All components (control) |
| `no_text` | Item title embedding removed |
| `no_svd` | SVD collaborative embedding removed |
| `no_numeric` | Numeric metadata removed |
| `no_cross_attn` | Cross-attention disabled |
| `unidirectional` | Only user-to-item attention |

In [ ]:
ABLATION_CONFIGS = {
    "full_model":      dict(use_text=True,  use_numeric=True,  use_svd=True,
                            use_cross_attn=True,  bidirectional=True),
    "no_text":         dict(use_text=False, use_numeric=True,  use_svd=True,
                            use_cross_attn=True,  bidirectional=True),
    "no_svd":          dict(use_text=True,  use_numeric=True,  use_svd=False,
                            use_cross_attn=True,  bidirectional=True),
    "no_numeric":      dict(use_text=True,  use_numeric=False, use_svd=True,
                            use_cross_attn=True,  bidirectional=True),
    "no_cross_attn":   dict(use_text=True,  use_numeric=True,  use_svd=True,
                            use_cross_attn=False, bidirectional=False),
    "unidirectional":  dict(use_text=True,  use_numeric=True,  use_svd=True,
                            use_cross_attn=True,  bidirectional=False),
}

print("\n" + "=" * 60)
print("  EXPERIMENT 4: Ablation Study (fold 0)")
print("=" * 60)

gkf_abl = GroupKFold(n_splits=NUM_FOLDS)
tr_abl_idx, te_abl_idx = next(iter(gkf_abl.split(indices, groups=user_ids_arr)))
train_abl = [interactions[i] for i in tr_abl_idx]
test_abl  = [interactions[i] for i in te_abl_idx]

ablation_results = {}
for name, flags in ABLATION_CONFIGS.items():
    print(f"\n-- Ablation: {name} --")
    result = run_fold(f"abl_{name}", train_abl, test_abl, **flags)
    ablation_results[name] = result

print("\n" + "=" * 70)
print("ABLATION SUMMARY")
print("=" * 70)
print(f"  {'Variant':<18s} {'MAE':>8s} {'RMSE':>8s} {'NDCG@10':>9s} {'HR@10':>8s}")
print(f"  {'-'*55}")
for name, res in ablation_results.items():
    print(f"  {name:<18s} {res['mae']:>8.4f} {res['rmse']:>8.4f} "
          f"{res.get(f'NDCG@{TOP_K}',0):>9.4f} {res.get(f'HR@{TOP_K}',0):>8.4f}")

---
## 17. Final Results Summary & Export

In [ ]:
print("\n" + "=" * 70)
print(f"  FINAL RESULTS: {DATASET.upper()}")
print("=" * 70)

print("\n-- Warm Evaluation (GroupKFold) --")
for metric in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
    vals = [r[metric] for r in warm_results]
    print(f"  {metric:>10s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

print("\n-- Baselines (fold 0) --")
for name, res in baseline_results.items():
    print(f"  {name:<12s}  MAE={res['mae']:.4f}  RMSE={res['rmse']:.4f}")
print(f"  {'BEST-Rec':<12s}  MAE={warm_results[0]['mae']:.4f}  RMSE={warm_results[0]['rmse']:.4f}")

print("\n-- Cold-Start --")
if cold_user_result:
    print(f"  Cold-User:  MAE={cold_user_result['mae']:.4f}  RMSE={cold_user_result['rmse']:.4f}")
if cold_item_result:
    print(f"  Cold-Item:  MAE={cold_item_result['mae']:.4f}  RMSE={cold_item_result['rmse']:.4f}")

print("\n-- Ablation --")
for name, res in ablation_results.items():
    print(f"  {name:<18s} MAE={res['mae']:.4f}  RMSE={res['rmse']:.4f}")

# ── Save all results ──
all_results = {
    "dataset":    DATASET,
    "warm":       warm_results,
    "baselines":  baseline_results,
    "cold_user":  cold_user_result,
    "cold_item":  cold_item_result,
    "ablation":   ablation_results,
    "config": {
        "HIDDEN_DIM": HIDDEN_DIM, "SVD_COMPONENTS": SVD_COMPONENTS,
        "LR": LR, "BATCH_SIZE": BATCH_SIZE, "EPOCHS": EPOCHS,
        "NUM_FOLDS": NUM_FOLDS, "SEED": SEED,
    },
}

with open(os.path.join(CACHE_DIR, "all_results.pkl"), "wb") as f:
    pickle.dump(all_results, f)

def jsonify(obj):
    if isinstance(obj, (np.floating,)):  return float(obj)
    if isinstance(obj, (np.integer,)):   return int(obj)
    if isinstance(obj, np.ndarray):      return obj.tolist()
    if isinstance(obj, dict):            return {k: jsonify(v) for k, v in obj.items()}
    if isinstance(obj, list):            return [jsonify(v) for v in obj]
    return obj

json_path = os.path.join(CACHE_DIR, "all_results.json")
with open(json_path, "w") as f:
    json.dump(jsonify(all_results), f, indent=2)

print(f"\nResults saved to {CACHE_DIR}/all_results.json")
print(f"Results saved to {CACHE_DIR}/all_results.pkl")

---
## 18. Running on Other Datasets

Change `DATASET` in cell 2 and **Restart & Run All**:

```python
DATASET = "fashion"       # or "books", "instruments"
```

Each dataset caches independently under `cache/<dataset>/`.

### Clearing Cache
```python
import shutil
shutil.rmtree("cache/beauty")  # forces full recompute
```